# Schema Migration: Old → New Column Names

Renames columns in old-schema joblib/CSV files to match the 202607+ schema:

| Old | New |
|-----|-----|
| `channel` | `model` |
| `model` | `estimator` |
| `base_model` | `base_estimator` |

Run cells in order. Originals are backed up to `data/output/backup/` before any files are overwritten.

In [1]:
import shutil
import sys
from pathlib import Path

import joblib
import pandas as pd

# Add project root to path so joblib can deserialize custom model classes
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import functions.lr_models  # noqa: F401

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
BACKUP_DIR = OUTPUT_DIR / "backup"

RENAME_MAP = {
    "channel": "model",
    "model": "estimator",
    "base_model": "base_estimator",
}

print(f"Output dir : {OUTPUT_DIR}")
print(f"Backup dir : {BACKUP_DIR}")


Output dir : /Users/jeff.parks/Dev/revenue-forecasting/data/output
Backup dir : /Users/jeff.parks/Dev/revenue-forecasting/data/output/backup


## 1. Discovery
Identify which joblib files use the old schema (contain a `channel` column).

In [2]:
all_joblins = sorted(OUTPUT_DIR.glob("models_*.joblib"))

needs_migration = []
already_new = []

for path in all_joblins:
    df = joblib.load(path)
    if "channel" in df.columns:
        needs_migration.append(path)
    else:
        already_new.append(path)

print(f"{'File':<40} {'Schema'}")
print("-" * 55)
for p in needs_migration:
    print(f"{p.name:<40} OLD — needs migration")
for p in already_new:
    print(f"{p.name:<40} NEW — skip")

print(f"\n{len(needs_migration)} file(s) will be migrated, {len(already_new)} already up to date.")

File                                     Schema
-------------------------------------------------------
models_202603_3P.joblib                  OLD — needs migration
models_202603_reflows.joblib             OLD — needs migration
models_202604_1P.joblib                  OLD — needs migration
models_202604_2P.joblib                  OLD — needs migration
models_202604_3P.joblib                  OLD — needs migration
models_202604_BRFS_flat.joblib           OLD — needs migration
models_202604_mmm.joblib                 OLD — needs migration
models_202605_1P.joblib                  OLD — needs migration
models_202605_2P.joblib                  OLD — needs migration
models_202605_3P.joblib                  OLD — needs migration
models_202606_1P.joblib                  OLD — needs migration
models_202606_2P.joblib                  NEW — skip
models_202607_1P.joblib                  NEW — skip
models_202607_2P.joblib                  NEW — skip

11 file(s) will be migrated, 3 already up to d

## 2. Preview (dry run)
Show the before/after columns for the first file that needs migration. No files are written.

In [3]:
if not needs_migration:
    print("Nothing to preview — all files are already on the new schema.")
else:
    sample_path = needs_migration[0]
    df_sample = joblib.load(sample_path)
    cols_before = [c for c in df_sample.columns if c != "model_obj"]
    cols_after = [RENAME_MAP.get(c, c) for c in cols_before]

    preview = pd.DataFrame({"Before": cols_before, "After": cols_after})
    changed = preview[preview["Before"] != preview["After"]]

    print(f"Preview for: {sample_path.name}")
    print(f"\nColumns being renamed ({len(changed)}):")
    display(changed.reset_index(drop=True))
    print(f"\nColumns unchanged: {list(preview[preview['Before'] == preview['After']]['Before'])}")

Preview for: models_202603_3P.joblib

Columns being renamed (3):


,Before,After
0,channel,model
1,model,estimator
2,base_model,base_estimator



Columns unchanged: ['brand', 'response', 'params', 'R2', 'R2_CV', 'n_obs', 'n_outliers', 'n_features', 'wls_k', 'SW_stat', 'SW_p', 'BP_stat', 'BP_p', 'DW', 'cooks_n', 'VIF']


## 3. Backup
Copy all `models_*.joblib` and `models_*.csv` files (old **and** new schema) to `data/output/backup/`.

> **Run this cell before proceeding.** The next cell overwrites files.

In [4]:
BACKUP_DIR.mkdir(exist_ok=True)

backed_up = []
for pattern in ("models_*.joblib", "models_*.csv"):
    for src in sorted(OUTPUT_DIR.glob(pattern)):
        dest = BACKUP_DIR / src.name
        shutil.copy2(src, dest)
        backed_up.append(src.name)

print(f"Backed up {len(backed_up)} files to {BACKUP_DIR.resolve()}:")
for name in backed_up:
    print(f"  {name}")

Backed up 28 files to /Users/jeff.parks/Dev/revenue-forecasting/data/output/backup:
  models_202603_3P.joblib
  models_202603_reflows.joblib
  models_202604_1P.joblib
  models_202604_2P.joblib
  models_202604_3P.joblib
  models_202604_BRFS_flat.joblib
  models_202604_mmm.joblib
  models_202605_1P.joblib
  models_202605_2P.joblib
  models_202605_3P.joblib
  models_202606_1P.joblib
  models_202606_2P.joblib
  models_202607_1P.joblib
  models_202607_2P.joblib
  models_202603_3P.csv
  models_202603_reflows.csv
  models_202604_1P.csv
  models_202604_2P.csv
  models_202604_3P.csv
  models_202604_BRFS_flat.csv
  models_202604_mmm.csv
  models_202605_1P.csv
  models_202605_2P.csv
  models_202605_3P.csv
  models_202606_1P.csv
  models_202606_2P.csv
  models_202607_1P.csv
  models_202607_2P.csv


## 4. Migration
Rename columns in each old-schema joblib and overwrite both the `.joblib` and companion `.csv`.

In [5]:
if not needs_migration:
    print("Nothing to migrate.")
else:
    for joblib_path in needs_migration:
        df = joblib.load(joblib_path)

        # Rename only columns that exist in this DataFrame
        actual_renames = {k: v for k, v in RENAME_MAP.items() if k in df.columns}
        df = df.rename(columns=actual_renames)

        # Save joblib
        joblib.dump(df, joblib_path)

        # Save companion CSV (exclude the model_obj column)
        csv_path = joblib_path.with_suffix(".csv")
        df.drop(columns=["model_obj"], errors="ignore").to_csv(csv_path, index=False)

        renamed_str = ", ".join(f"{k}→{v}" for k, v in actual_renames.items())
        print(f"✓  {joblib_path.name}  [{renamed_str}]")

    print(f"\nMigrated {len(needs_migration)} file(s).")

✓  models_202603_3P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202603_reflows.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202604_1P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202604_2P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202604_3P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202604_BRFS_flat.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202604_mmm.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202605_1P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202605_2P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202605_3P.joblib  [channel→model, model→estimator, base_model→base_estimator]
✓  models_202606_1P.joblib  [channel→model, model→estimator, base_model→base_estimator]

Migrated 11 file(s

## 5. Verify
Reload all joblib files and confirm no `channel` or `base_model` columns remain.

In [6]:
legacy_cols = {"channel", "base_model"}
issues = []

for path in sorted(OUTPUT_DIR.glob("models_*.joblib")):
    df = joblib.load(path)
    found = legacy_cols & set(df.columns)
    if found:
        issues.append((path.name, found))

if issues:
    print("WARNING — legacy columns still present:")
    for name, cols in issues:
        print(f"  {name}: {cols}")
else:
    all_files = sorted(OUTPUT_DIR.glob("models_*.joblib"))
    print(f"All {len(all_files)} file(s) verified — no legacy columns found.")
    for p in all_files:
        df = joblib.load(p)
        key_cols = [c for c in ["brand", "model", "estimator", "base_estimator", "response"] if c in df.columns]
        print(f"  {p.name}: {key_cols}")

All 14 file(s) verified — no legacy columns found.
  models_202603_3P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202603_reflows.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202604_1P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202604_2P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202604_3P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202604_BRFS_flat.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202604_mmm.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202605_1P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202605_2P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202605_3P.joblib: ['brand', 'model', 'estimator', 'base_estimator', 'response']
  models_202606_1P.joblib: ['brand', 'model', 